# Fitting wells

This notebook introduces the concept of fitting wells and how they are used in `gwrefpy`. The methodology originates from the description by its author [Strandanger (2024)](https://svenskageotekniskaforeningen.se/wp-content/uploads/Publikationer/SGF_Rapporter/2024_2_Akvifars_refmetod.pdf).

This notebook can be downloaded from the source code [here](https://github.com/andersretznerSGU/gwrefpy/blob/main/docs/user_guide/2_fitting_wells.ipynb).

To analyse deviations in groundwater level timeseries, the `gwrefpy` methodology relies on observation wells and reference wells. To check for deviations in an observation well, we fit its data to a reference well using regression. When fitting the two wells, we specify a calibration period. If data outside of the calibration period does not follow the fitted regression, a deviation has occurred. 

In this notebook, we will dive deeper into the fitting workflow and showcase this library's associated capabilities.

```{note}
See [this notebook on wells](https://github.com/andersretznerSGU/gwrefpy/blob/main/docs/user_guide/1_intro_gwrefpy.ipynb)
```


In [ ]:
import gwrefpy as gr
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
gr.set_log_level("ERROR")
plt.set_loglevel("ERROR")


## Data

In this notebook, we will work with data provided by `gwrefpy`. Let's load the model using its file name and plot the observation well and reference well separately.

In [ ]:
model = gr.Model("deviation_example.gwref")
model

In [ ]:
fig, (up, down) = plt.subplots(nrows=2, sharex=True)

for well in model.wells.ref():
    well.timeseries.plot(ax=up)
for well in model.wells.obs():
    well.timeseries.plot(ax=down)

up.set_title("obs wells")
down.set_title("ref wells")

up.legend()
down.legend()

fig.autofmt_xdate(rotation=45)

## Anatomy of a fit

Let's walk through the anatomy of a fit by fitting the observation well _12GIPGW_ to the reference well _45LOGW_. To fit wells within a model, we call `model.fit()` which has the following signature:

In [ ]:
help(model.fit)

The first two arguments determine which wells to use. 

The `offset` and `aggregation` arguments handle cases where the timeseries of the wells are not aligned, see [this notebook](). The wells in this model are sampled daily and cover the same period, which means we can use `"0D"` as the offset which also renders the `aggregation` argument obsolete.

The `p` argument controls which confidence level to use when evaluating the lower and upper bound from which to recognize deviations. We will use the default `p`-value of 0.05.

The `tmin` and `tmax` arguments control the period in which we want to perform the fit. This is an important decision that can only be inferred from prior knowledge of the wells. We want to select this period such that no (or negligble) external factors are affecting head in the observation well. In this example, we will assume that data until 2022 is unaffected and can be used for fitting a regression.

```{important}
The reference well needs to be unaffected by external factors during its entire time period.
```

Below, we introduce the linear regression fit as demonstrated in [Strandanger (2024)](https://svenskageotekniskaforeningen.se/wp-content/uploads/Publikationer/SGF_Rapporter/2024_2_Akvifars_refmetod.pdf). Other fitting methods are still being introduced to `gwrefpy` and will be documented in future releases. Look below, under _Other fitting methods_, for more information on current options. You can also refer to [this notebook](https://github.com/andersretznerSGU/gwrefpy/blob/main/docs/user_guide/7_fitting_methods.ipynb).

Let's do this!

In [ ]:
fit = model.fit(
    obs_well="19W110U",
    ref_well="19W100U",
    offset="0D",
    tmax="2022"
)

By default when calling `fit()`, a summary of the fitted regression is displayed as seen above.

**TODO: fix ahead v2 to align with new plotting behavior**

Let's plot the resulting fit to verify everything is OK. We can use the `plot` method and pass all available fits knowing it will only contain the fit we just created. We also evaluate the fit performance by testing it against the reference well data using the `test_fit()` method. Here, we specify the same offset and use the tmin equal to the calibration period end.

In [ ]:
model.plot_fitmethod()
stderr, rmse = fit.test_fit(model.ref_wells[0].timeseries, offset="0D", tmin="2022")
print("Fit performance on reference well data after calibration period:")
print(f"Standard error: {stderr:.3f}")
print(f"RMSE: {rmse:.3f}")

We can see that for the training data, which is data used to fit the regression, our fit seems OK. Observed heads outside of this period show a different behavior - one might suspect a deviation!

## Analysing deviations

Evaluating deviation is a key `gwrefpy` feature! Below, we plot the resulting fit and transformed head, along with the lower and upper confidence limit based on our `p` value (0.05) and the fit. 

Let's first obtain our fit assuming we didn't catch when creating the fit or if our model had multiple fits. Fits can be filtered and removed just like a model's wells.

Below, we obtain our fit using it's name which defaults to *{obs}\~{ref}:{method}*. We could also have a used a filter as the second code line shows.

In [ ]:
fit = model.fits.get(name="19W110U~19W100U:Linear")
fit = model.fits.get(obs="19W110U")  # results in same fit as above (in this case)
fit

Let's also plot the resulting deviations.

**TODO: align with new plotting behavior for v2**

```{tip}
Below, we generate the plot by hand to showcase how to retrieve the fitted timeseries and its associated upper and lower bound. If you want to quickly review a fit and deviations, use `model.plot_fits()`.
```

In [ ]:
fitted_ref = fit.get_fit_timeseries()
upper_bound = fit.get_upper_confidence_bound()
lower_bound = fit.get_lower_confidence_bound()

fig, ax = plt.subplots()

_ = ax.fill_between(fitted_ref.index, y1=lower_bound, y2=upper_bound, alpha=0.5)
_ = ax.axvline(pd.Timestamp("2022"), color="black", linewidth=2, label="Calibration stop")

_ = fitted_ref.plot(ax=ax, label="19W100U (fitted)")
_ = obs = model.get_wells("19W110U")
_ = obs.timeseries.plot(ax=ax, label="19W110U")
_ = ax.legend(loc="upper left")

Seems like the observation well (19W110U) dips below the lower bound during 2023 and 2024. Let's zoom into this period and plot the timesteps where this occurs with a different color after retrivieing them with the `get_outliers()` method.

In [ ]:
fig, ax = plt.subplots()

ax.fill_between(fitted_ref.index, y1=lower_bound, y2=upper_bound, alpha=0.5)

fitted_ref.loc["2022":].plot(ax=ax, label="19W100U (fitted)")
obs.timeseries.loc["2022":].plot(ax=ax, label="19W110U")
_ = ax.legend()

## Generate multiple fits at once

With `gwrefpy`, you can fit as many reference wells as you like to a single observation well. This is useful if you have many wells to choose from, and would like to rank their fit performance.

Below, we load an example model with many observation and reference wells and clear its existing fits.

In [ ]:
large_model = gr.Model("large_example.gwref")
large_model.fits.clear()

print(
    f"There are {len(large_model.wells)} wells in this model "
    f"and, since just recently, {len(large_model.fits)} fits."
)

We choose a observation well and then run `best_fit()`, which will default to perform a fit for all available reference wells.

This project many reference wells, so we select only the top 3 fits with `remove_fits_by_n()` and then plot the resulting regression fit by fit. Notice we can chain things like `keep_best_n()` and a `filter()`.

In [ ]:
_ = large_model.best_fit(obs_well="01OBS", offset="0D")
top_three_fits = large_model.fits.keep_best(3, by=None).filter(obs="01OBS")

fig, axs = plt.subplots(ncols=3, sharey=True, figsize=(10,4))
for ax, fit in zip(axs, top_three_fits):
    _ = large_model.plot_fitmethod(fit, ax=ax)
    legend = ax.get_legend()
    legend.set_bbox_to_anchor((0.5, -0.5))
    legend.set_loc("upper center")

This is a great way to evaluate many reference wells for a single observation well!

# Other fitting methods
In addition to linear regression, `gwrefpy` also supports n-polynomial fitting and Chebyshev polynomial fitting. These methods are useful when the fitting relationship between the observation well and reference well requires a higher degree of freedom. Below, we fit the same wells as before, but this time using an n-polynomial of degree 5.

```{tip}
These methods are described in more detail in the [Fitting methods](https://sgfsweden.github.io/gwrefpy/user_guide/7_fitting_methods.html) section of the user guide.
```

In [ ]:
model.fit(
    obs_well="19W110U",
    ref_well="19W100U",
    offset="0D",
    tmax="2022",
    method=gr.NPoly(degree=5),
)

Let's access the associated fit an plot the diagnostics plot.

**TODO: align with new plotting behavior for v2**

In [ ]:
fit_npoly = model.fits.get(obs="19W110U", method="npoly")
fit_linear = model.fits.get(obs="19W110U", method="linear")

difference = fit_npoly.get_fit_timeseries() - fit_linear.get_fit_timeseries()

fig, ax = plt.subplots()

_ = ax.plot(
    difference.index,
    difference,
    label="Difference in fit",
    color="C0",
)
_ = ax.set_xlabel("Date")
_ = ax.set_ylabel("Fitted value difference (m)")

We can see that the two fitting are very similar for this specific case. Depending on the application, one might be preferred over the other.

Happy fitting!